# EU AI Act: Practical Compliance for AI Engineers

The EU AI Act entered into force in August 2024 and is the world's first comprehensive AI regulation. It takes a risk-based approach: the higher the risk an AI system poses to people, the stricter the obligations. Implementation is phased through 2026-2027, with bans on unacceptable-risk systems taking effect first and high-risk obligations rolling in over time.

This notebook covers:
1. Risk Classification Framework
2. High-Risk System Obligations
3. Technical Implementation
4. NIST AI RMF Alignment
5. Practical Compliance Workflow

In [1]:
import pandas as pd
import json
from datetime import datetime
from pydantic import BaseModel, Field
from typing import List, Optional, Dict, Any
from enum import Enum

## Section 1: Risk Classification Framework

The EU AI Act divides AI systems into four risk tiers. Each tier carries different legal obligations.

### Unacceptable Risk (Banned)
These systems are prohibited outright:
- Real-time remote biometric identification in public spaces (with narrow law-enforcement exceptions)
- Social scoring by governments or public bodies
- Subliminal manipulation techniques that exploit psychological weaknesses
- AI that exploits children or persons with disabilities
- Predictive policing based solely on profiling

### High Risk (Heavily Regulated)
These systems must meet strict requirements before deployment:
- **Biometric systems**: remote identification, categorization, emotion recognition
- **Critical infrastructure**: power grids, water systems, transportation
- **Education**: student assessment, access decisions
- **Employment**: CV screening, hiring, promotion, termination
- **Essential services**: credit scoring, insurance underwriting
- **Law enforcement**: crime prediction, evidence reliability, suspect profiling
- **Border control**: risk assessment, document verification
- **Administration of justice**: judicial decision support

### Limited Risk (Transparency Requirements)
These systems must be transparent about their AI nature:
- Chatbots and conversational AI must identify as AI to users
- Deepfakes must be labeled as AI-generated
- Emotion recognition outputs must be disclosed

### Minimal Risk (No Special Rules)
The vast majority of AI applications fall here:
- Spam filters
- AI in video games
- AI-powered recommendation engines (most)
- Inventory management AI

In [2]:
class RiskTier(str, Enum):
    UNACCEPTABLE = "unacceptable"
    HIGH = "high"
    LIMITED = "limited"
    MINIMAL = "minimal"


# Sector and use-case mappings to risk tier
HIGH_RISK_SECTORS = {
    "employment", "hiring", "hr", "recruitment",
    "credit", "finance", "insurance", "lending",
    "education", "assessment", "admissions",
    "healthcare", "medical", "clinical",
    "critical_infrastructure", "energy", "water", "transport",
    "law_enforcement", "police", "justice", "courts",
    "border_control", "immigration", "asylum",
    "biometric", "facial_recognition"
}

BANNED_PATTERNS = [
    "social scoring",
    "social credit",
    "real-time biometric surveillance",
    "real time biometric",
    "subliminal manipulation",
    "predictive policing profiling",
    "exploit children",
    "exploit vulnerability"
]

LIMITED_RISK_PATTERNS = [
    "chatbot",
    "conversational ai",
    "virtual assistant",
    "deepfake",
    "synthetic media",
    "emotion recognition"
]


class RiskClassificationResult(BaseModel):
    risk_tier: RiskTier
    rationale: str
    obligations: List[str]
    recommended_actions: List[str]


class RiskClassifier:
    """Classify an AI system under the EU AI Act risk tiers."""

    def classify_system(
        self,
        description: str,
        sector: str,
        use_case: str
    ) -> RiskClassificationResult:
        description_lower = description.lower()
        sector_lower = sector.lower()
        use_case_lower = use_case.lower()
        combined = f"{description_lower} {sector_lower} {use_case_lower}"

        # Check for banned systems first
        for pattern in BANNED_PATTERNS:
            if pattern in combined:
                return RiskClassificationResult(
                    risk_tier=RiskTier.UNACCEPTABLE,
                    rationale=f"System matches banned pattern: '{pattern}'",
                    obligations=["This system is PROHIBITED under the EU AI Act."],
                    recommended_actions=[
                        "Do not deploy this system in the EU.",
                        "Consult legal counsel immediately.",
                        "Redesign the system to remove the prohibited capability."
                    ]
                )

        # Check for high-risk sectors
        sector_words = set(sector_lower.replace("-", "_").split())
        if sector_words & HIGH_RISK_SECTORS:
            return RiskClassificationResult(
                risk_tier=RiskTier.HIGH,
                rationale=f"Sector '{sector}' falls under Annex III high-risk categories.",
                obligations=[
                    "Technical documentation (Article 11)",
                    "Risk management system (Article 9)",
                    "Data governance and bias testing (Article 10)",
                    "Automatic logging of decisions (Article 12)",
                    "Transparency and explainability (Article 13)",
                    "Human oversight capability (Article 14)",
                    "Accuracy, robustness, and cybersecurity (Article 15)",
                    "Conformity assessment before deployment",
                    "Registration in EU database",
                    "Post-market monitoring"
                ],
                recommended_actions=[
                    "Conduct a full compliance audit before deployment.",
                    "Implement AuditLogger for all AI decisions.",
                    "Add SHAP explainability wrapper.",
                    "Run demographic bias tests on training data.",
                    "Implement human override capability.",
                    "Register system in EU AI database before deployment."
                ]
            )

        # Check for limited risk (transparency requirements)
        for pattern in LIMITED_RISK_PATTERNS:
            if pattern in combined:
                return RiskClassificationResult(
                    risk_tier=RiskTier.LIMITED,
                    rationale=f"System matches limited-risk pattern: '{pattern}'",
                    obligations=[
                        "Disclose AI nature to users (users must know they are interacting with AI)",
                        "Label AI-generated content as such"
                    ],
                    recommended_actions=[
                        "Add clear 'You are speaking with an AI' notice in the UI.",
                        "Label any generated content with an AI disclosure badge."
                    ]
                )

        # Default: minimal risk
        return RiskClassificationResult(
            risk_tier=RiskTier.MINIMAL,
            rationale="System does not match any higher-risk category.",
            obligations=["No mandatory EU AI Act obligations at this time."],
            recommended_actions=[
                "Voluntarily follow AI ethics best practices.",
                "Reassess if use-case or sector changes.",
                "Monitor for future regulatory updates."
            ]
        )


# Demo the classifier
classifier = RiskClassifier()

test_cases = [
    {"description": "CV screening tool that ranks job applicants", "sector": "employment", "use_case": "hiring"},
    {"description": "Customer service chatbot", "sector": "retail", "use_case": "support"},
    {"description": "Government social scoring system", "sector": "government", "use_case": "social scoring"},
    {"description": "Spam filter for email", "sector": "productivity", "use_case": "email filtering"},
    {"description": "Credit risk model for loan applications", "sector": "credit", "use_case": "lending"}
]

for tc in test_cases:
    result = classifier.classify_system(tc["description"], tc["sector"], tc["use_case"])
    print(f"System: {tc['description']}")
    print(f"  Risk Tier: {result.risk_tier.value.upper()}")
    print(f"  Rationale: {result.rationale}")
    print()

System: CV screening tool that ranks job applicants
  Risk Tier: HIGH
  Rationale: Sector 'employment' falls under Annex III high-risk categories.

System: Customer service chatbot
  Risk Tier: LIMITED
  Rationale: System matches limited-risk pattern: 'chatbot'

System: Government social scoring system
  Risk Tier: UNACCEPTABLE
  Rationale: System matches banned pattern: 'social scoring'

System: Spam filter for email
  Risk Tier: MINIMAL
  Rationale: System does not match any higher-risk category.

System: Credit risk model for loan applications
  Risk Tier: HIGH
  Rationale: Sector 'credit' falls under Annex III high-risk categories.



## Section 2: High-Risk System Obligations (for Engineers)

If your system is classified as high-risk, the EU AI Act imposes these specific obligations. Each has concrete engineering implications:

| Article | Requirement | Engineering Task |
|---------|-------------|------------------|
| Art. 9 | Risk management system | Identify and document risks; test mitigations |
| Art. 10 | Data governance | Document training data sources, quality measures, bias tests |
| Art. 11 | Technical documentation | Maintain full technical spec (model card ++) |
| Art. 12 | Logging | Log every AI decision: inputs, outputs, timestamp, model version |
| Art. 13 | Transparency | User notices; explainability for individual decisions |
| Art. 14 | Human oversight | Implement override capability; design for human review |
| Art. 15 | Accuracy/robustness | Document accuracy metrics; test adversarial robustness |

Log retention: **10 years** for decisions that affect individual rights.

In [3]:
class ComplianceItem(BaseModel):
    article: str
    requirement: str
    completed: bool = False
    notes: str = ""
    completion_date: Optional[str] = None


class ComplianceChecklist(BaseModel):
    system_name: str
    system_version: str
    assessment_date: str = Field(default_factory=lambda: datetime.now().isoformat())
    items: List[ComplianceItem] = Field(default_factory=list)

    @classmethod
    def for_high_risk_system(cls, system_name: str, system_version: str) -> "ComplianceChecklist":
        items = [
            ComplianceItem(article="Art. 9", requirement="Risk management system: documented risk identification, analysis, evaluation, and control measures"),
            ComplianceItem(article="Art. 10", requirement="Data governance: training data documented, quality measures in place, bias tested"),
            ComplianceItem(article="Art. 11", requirement="Technical documentation: full model card, architecture, training procedure, performance metrics"),
            ComplianceItem(article="Art. 12", requirement="Automatic logging: every AI decision logged with timestamp, inputs, outputs, model version"),
            ComplianceItem(article="Art. 12", requirement="Log retention: logs stored for minimum 10 years"),
            ComplianceItem(article="Art. 13", requirement="Transparency: users notified they are interacting with AI"),
            ComplianceItem(article="Art. 13", requirement="Explainability: explanations available for individual AI decisions"),
            ComplianceItem(article="Art. 14", requirement="Human oversight: override capability implemented and tested"),
            ComplianceItem(article="Art. 15", requirement="Accuracy: performance metrics documented on representative test set"),
            ComplianceItem(article="Art. 15", requirement="Robustness: adversarial robustness tested"),
            ComplianceItem(article="Art. 15", requirement="Cybersecurity: model and API secured against adversarial attacks"),
            ComplianceItem(article="Annex IV", requirement="Conformity assessment completed"),
            ComplianceItem(article="Art. 51", requirement="System registered in EU AI database before deployment")
        ]
        return cls(system_name=system_name, system_version=system_version, items=items)

    def mark_complete(self, article: str, notes: str = "") -> None:
        for item in self.items:
            if item.article == article and not item.completed:
                item.completed = True
                item.notes = notes
                item.completion_date = datetime.now().isoformat()
                break

    def completion_rate(self) -> float:
        if not self.items:
            return 0.0
        return sum(1 for i in self.items if i.completed) / len(self.items)

    def print_status(self) -> None:
        print(f"Compliance Checklist: {self.system_name} v{self.system_version}")
        print(f"Assessment Date: {self.assessment_date[:10]}")
        print(f"Completion: {self.completion_rate():.0%}")
        print("-" * 70)
        for item in self.items:
            status = "[X]" if item.completed else "[ ]"
            print(f"{status} {item.article}: {item.requirement[:60]}")
            if item.notes:
                print(f"     Notes: {item.notes}")


# Demo
checklist = ComplianceChecklist.for_high_risk_system("HiringScreenerV2", "2.1.0")
checklist.mark_complete("Art. 11", notes="Model card v2.1 published to internal wiki")
checklist.mark_complete("Art. 12", notes="AuditLogger deployed to production 2024-11-01")
checklist.mark_complete("Art. 14", notes="HR manager override button added to review interface")
checklist.print_status()

Compliance Checklist: HiringScreenerV2 v2.1.0
Assessment Date: 2026-06-29
Completion: 23%
----------------------------------------------------------------------
[ ] Art. 9: Risk management system: documented risk identification, anal
[ ] Art. 10: Data governance: training data documented, quality measures 
[X] Art. 11: Technical documentation: full model card, architecture, trai
     Notes: Model card v2.1 published to internal wiki
[X] Art. 12: Automatic logging: every AI decision logged with timestamp, 
     Notes: AuditLogger deployed to production 2024-11-01
[ ] Art. 12: Log retention: logs stored for minimum 10 years
[ ] Art. 13: Transparency: users notified they are interacting with AI
[ ] Art. 13: Explainability: explanations available for individual AI dec
[X] Art. 14: Human oversight: override capability implemented and tested
     Notes: HR manager override button added to review interface
[ ] Art. 15: Accuracy: performance metrics documented on representative t
[ ] Art. 15: 

## Section 3: Technical Implementation

High-risk system compliance requires three concrete engineering components:

1. **Audit Trail Logging**: Every AI decision must be logged permanently
2. **Explainability**: Each decision must have an explanation available
3. **Bias Testing**: Demographic bias must be tested before deployment

### 3.1 Audit Trail Logger

In [4]:
import hashlib
import uuid


class AuditRecord(BaseModel):
    record_id: str = Field(default_factory=lambda: str(uuid.uuid4()))
    timestamp: str = Field(default_factory=lambda: datetime.now().isoformat())
    model_name: str
    model_version: str
    input_hash: str  # Hash of input data (not raw data, for privacy)
    decision: Any
    confidence: Optional[float] = None
    human_reviewed: bool = False
    human_decision: Optional[Any] = None
    override_reason: Optional[str] = None


class AuditLogger:
    """EU AI Act compliant audit logger. Stores decisions for 10-year retention."""

    def __init__(self, model_name: str, model_version: str, storage_path: str = "audit_log.jsonl"):
        self.model_name = model_name
        self.model_version = model_version
        self.storage_path = storage_path
        self._records: List[AuditRecord] = []

    def _hash_input(self, input_data: Any) -> str:
        """Hash inputs for privacy-preserving audit trail."""
        serialized = json.dumps(input_data, sort_keys=True, default=str)
        return hashlib.sha256(serialized.encode()).hexdigest()[:16]

    def log_decision(
        self,
        input_data: Any,
        decision: Any,
        confidence: Optional[float] = None
    ) -> AuditRecord:
        record = AuditRecord(
            model_name=self.model_name,
            model_version=self.model_version,
            input_hash=self._hash_input(input_data),
            decision=decision,
            confidence=confidence
        )
        self._records.append(record)
        # In production: write to append-only storage (WORM, immutable blob)
        return record

    def log_human_override(
        self,
        record_id: str,
        human_decision: Any,
        override_reason: str
    ) -> None:
        for record in self._records:
            if record.record_id == record_id:
                record.human_reviewed = True
                record.human_decision = human_decision
                record.override_reason = override_reason
                break

    def get_recent_decisions(self, n: int = 5) -> List[Dict]:
        return [r.model_dump() for r in self._records[-n:]]

    def summary_stats(self) -> Dict:
        total = len(self._records)
        overrides = sum(1 for r in self._records if r.human_reviewed)
        return {
            "total_decisions": total,
            "human_overrides": overrides,
            "override_rate": overrides / total if total > 0 else 0
        }


# Demo
logger = AuditLogger(model_name="CVScreener", model_version="2.1.0")

candidates = [
    {"candidate_id": "C001", "years_experience": 5, "education": "MSc"},
    {"candidate_id": "C002", "years_experience": 2, "education": "BSc"},
    {"candidate_id": "C003", "years_experience": 8, "education": "PhD"}
]

decisions = ["ADVANCE", "REJECT", "ADVANCE"]
records = []
for candidate, decision in zip(candidates, decisions):
    record = logger.log_decision(candidate, decision, confidence=0.87)
    records.append(record)
    print(f"Logged: {candidate['candidate_id']} -> {decision} (ID: {record.record_id[:8]}...)")

# Human overrides C002's rejection
logger.log_human_override(records[1].record_id, "ADVANCE", "Strong portfolio despite fewer years")
print("\nSummary Stats:", logger.summary_stats())

Logged: C001 -> ADVANCE (ID: 445923f5...)
Logged: C002 -> REJECT (ID: f72c16ce...)
Logged: C003 -> ADVANCE (ID: b1072d7e...)

Summary Stats: {'total_decisions': 3, 'human_overrides': 1, 'override_rate': 0.3333333333333333}


### 3.2 Explainability Wrapper

Article 13 requires that high-risk AI systems provide explanations for individual decisions. SHAP (SHapley Additive exPlanations) is the standard approach for tabular models.

In [5]:
class FeatureContribution(BaseModel):
    feature_name: str
    feature_value: Any
    contribution: float  # Positive = pushed toward positive outcome
    importance_rank: int


class Explanation(BaseModel):
    decision: Any
    confidence: float
    top_factors: List[FeatureContribution]
    plain_language: str


class ExplainabilityWrapper:
    """Wraps a model to add SHAP-style explanations for each decision.
    
    In production: use shap.TreeExplainer or shap.LinearExplainer with your actual model.
    This demo shows the interface and output format required for EU AI Act compliance.
    """

    def __init__(self, model_fn, feature_names: List[str]):
        self.model_fn = model_fn
        self.feature_names = feature_names

    def predict_with_explanation(
        self,
        input_data: Dict[str, Any]
    ) -> Explanation:
        decision, confidence = self.model_fn(input_data)

        # In production: compute actual SHAP values
        # shap_values = explainer.shap_values(input_array)
        # Here we simulate contributions for demonstration
        simulated_contributions = {
            feat: (input_data.get(feat, 0) * 0.1 + (0.05 if i % 2 == 0 else -0.03))
            for i, feat in enumerate(self.feature_names)
            if feat in input_data
        }

        sorted_features = sorted(
            simulated_contributions.items(),
            key=lambda x: abs(x[1]),
            reverse=True
        )

        top_factors = [
            FeatureContribution(
                feature_name=name,
                feature_value=input_data[name],
                contribution=round(contrib, 3),
                importance_rank=rank + 1
            )
            for rank, (name, contrib) in enumerate(sorted_features[:3])
        ]

        # Generate plain-language explanation
        top_positive = [f for f in top_factors if f.contribution > 0]
        top_negative = [f for f in top_factors if f.contribution < 0]
        factors_text = ", ".join(f.feature_name.replace("_", " ") for f in top_positive[:2])
        against_text = ", ".join(f.feature_name.replace("_", " ") for f in top_negative[:1])

        plain = f"Decision: {decision} (confidence {confidence:.0%}). "
        if factors_text:
            plain += f"Key factors supporting this decision: {factors_text}. "
        if against_text:
            plain += f"Factors working against: {against_text}."

        return Explanation(
            decision=decision,
            confidence=confidence,
            top_factors=top_factors,
            plain_language=plain
        )


# Demo model
def simple_credit_model(features: Dict) -> tuple:
    score = features.get("credit_score", 600) / 850
    income_factor = min(features.get("annual_income", 30000) / 100000, 1.0)
    combined = 0.6 * score + 0.4 * income_factor
    decision = "APPROVE" if combined > 0.55 else "DECLINE"
    return decision, combined


wrapper = ExplainabilityWrapper(
    model_fn=simple_credit_model,
    feature_names=["credit_score", "annual_income", "debt_to_income", "employment_years"]
)

applicant = {"credit_score": 720, "annual_income": 75000, "debt_to_income": 0.28, "employment_years": 5}
explanation = wrapper.predict_with_explanation(applicant)

print("CREDIT DECISION WITH EXPLANATION")
print("-" * 50)
print(f"Decision: {explanation.decision}")
print(f"Confidence: {explanation.confidence:.1%}")
print("\nTop Contributing Factors:")
for factor in explanation.top_factors:
    direction = "(+)" if factor.contribution > 0 else "(-)"
    print(f"  #{factor.importance_rank} {direction} {factor.feature_name}: {factor.feature_value} | contribution: {factor.contribution}")
print(f"\nPlain Language: {explanation.plain_language}")

CREDIT DECISION WITH EXPLANATION
--------------------------------------------------
Decision: APPROVE
Confidence: 80.8%

Top Contributing Factors:
  #1 (+) annual_income: 75000 | contribution: 7499.97
  #2 (+) credit_score: 720 | contribution: 72.05
  #3 (+) employment_years: 5 | contribution: 0.47

Plain Language: Decision: APPROVE (confidence 81%). Key factors supporting this decision: annual income, credit score. 


### 3.3 Bias Testing Suite

Article 10 requires training data quality measures including bias testing. Demographic parity is the most commonly required metric: the positive outcome rate should not differ significantly across protected groups.

In [6]:
class GroupBiasMetrics(BaseModel):
    group: str
    n: int
    positive_rate: float
    true_positive_rate: Optional[float] = None  # Recall / equal opportunity
    false_positive_rate: Optional[float] = None


class BiasTestResult(BaseModel):
    attribute: str
    groups: List[GroupBiasMetrics]
    demographic_parity_gap: float
    passes_threshold: bool
    threshold: float = 0.10  # Max allowed gap in positive rates
    recommendation: str


class BiasTestSuite:
    """Test for demographic bias across protected attributes before deployment."""

    def __init__(self, threshold: float = 0.10):
        self.threshold = threshold
        self.results: List[BiasTestResult] = []

    def test_demographic_parity(
        self,
        predictions: List[int],
        group_labels: List[str],
        attribute_name: str,
        ground_truth: Optional[List[int]] = None
    ) -> BiasTestResult:
        pred_df = pd.DataFrame({
            "prediction": predictions,
            "group": group_labels,
            "truth": ground_truth if ground_truth else [None] * len(predictions)
        })

        group_metrics = []
        for group_name, group_df in pred_df.groupby("group"):
            positive_rate = group_df["prediction"].mean()
            tpr = None
            fpr = None
            if ground_truth:
                tp = ((group_df["prediction"] == 1) & (group_df["truth"] == 1)).sum()
                fn = ((group_df["prediction"] == 0) & (group_df["truth"] == 1)).sum()
                fp = ((group_df["prediction"] == 1) & (group_df["truth"] == 0)).sum()
                tn = ((group_df["prediction"] == 0) & (group_df["truth"] == 0)).sum()
                tpr = tp / (tp + fn) if (tp + fn) > 0 else None
                fpr = fp / (fp + tn) if (fp + tn) > 0 else None

            group_metrics.append(GroupBiasMetrics(
                group=str(group_name),
                n=len(group_df),
                positive_rate=round(positive_rate, 3),
                true_positive_rate=round(tpr, 3) if tpr is not None else None,
                false_positive_rate=round(fpr, 3) if fpr is not None else None
            ))

        rates = [m.positive_rate for m in group_metrics]
        parity_gap = max(rates) - min(rates)
        passes = parity_gap <= self.threshold

        recommendation = (
            f"PASS: Demographic parity gap ({parity_gap:.3f}) within threshold ({self.threshold}). Safe to deploy."
            if passes else
            f"FAIL: Demographic parity gap ({parity_gap:.3f}) exceeds threshold ({self.threshold}). "
            f"Re-evaluate training data and model for bias before deployment."
        )

        result = BiasTestResult(
            attribute=attribute_name,
            groups=group_metrics,
            demographic_parity_gap=round(parity_gap, 3),
            passes_threshold=passes,
            threshold=self.threshold,
            recommendation=recommendation
        )
        self.results.append(result)
        return result

    def print_report(self) -> None:
        for result in self.results:
            print(f"Bias Test: {result.attribute}")
            print(f"  Status: {'PASS' if result.passes_threshold else 'FAIL'}")
            print(f"  Parity Gap: {result.demographic_parity_gap:.3f} (threshold: {result.threshold})")
            for g in result.groups:
                tpr_str = f" | TPR: {g.true_positive_rate:.3f}" if g.true_positive_rate else ""
                print(f"  [{g.group}] n={g.n}, positive_rate={g.positive_rate:.3f}{tpr_str}")
            print(f"  Recommendation: {result.recommendation}")
            print()


# Demo
import numpy as np
rng = np.random.default_rng(42)

# Simulate biased hiring model output
n = 200
groups = ["Group A"] * 100 + ["Group B"] * 100
# Group A: 65% advance rate; Group B: 42% advance rate -- biased!
preds = list(rng.binomial(1, 0.65, 100)) + list(rng.binomial(1, 0.42, 100))
truth = list(rng.binomial(1, 0.55, 100)) + list(rng.binomial(1, 0.55, 100))

suite = BiasTestSuite(threshold=0.10)
result = suite.test_demographic_parity(preds, groups, "Gender", ground_truth=truth)
suite.print_report()

Bias Test: Gender
  Status: FAIL
  Parity Gap: 0.220 (threshold: 0.1)
  [Group A] n=100, positive_rate=0.640 | TPR: 0.589
  [Group B] n=100, positive_rate=0.420 | TPR: 0.423
  Recommendation: FAIL: Demographic parity gap (0.220) exceeds threshold (0.1). Re-evaluate training data and model for bias before deployment.



## Section 4: NIST AI RMF Alignment

The NIST AI Risk Management Framework (AI RMF 1.0) is the US government's voluntary framework for AI risk management. It maps well to EU AI Act requirements and provides a practical engineering structure.

| NIST Core Function | What It Means | EU AI Act Mapping |
|-------------------|---------------|--------------------|
| **GOVERN** | Policies, accountability, culture | Conformity assessment, registration |
| **MAP** | Categorize AI risks in context | Risk classification, use-case analysis |
| **MEASURE** | Quantify risks with metrics | Bias testing, accuracy metrics, robustness |
| **MANAGE** | Control and mitigate risks | Logging, explainability, human oversight |

In [7]:
nist_rmf_checklist = {
    "GOVERN": [
        "AI risk management policies documented",
        "Accountability roles assigned (AI Officer or equivalent)",
        "Organizational AI risk tolerance defined",
        "Third-party AI system inventory maintained"
    ],
    "MAP": [
        "AI system purpose and context documented",
        "Risk classification completed (EU AI Act tier)",
        "Affected populations and potential harms identified",
        "Intended vs. actual deployment context verified"
    ],
    "MEASURE": [
        "Accuracy metrics benchmarked on representative data",
        "Demographic bias tests run on all protected attributes",
        "Adversarial robustness evaluated",
        "Human-AI team performance measured"
    ],
    "MANAGE": [
        "Automatic audit logging deployed",
        "Explainability wrapper operational",
        "Human override capability tested",
        "Incident response plan for AI failures",
        "Post-market monitoring dashboard live"
    ]
}

print("NIST AI RMF Implementation Checklist")
print("=" * 60)
for function, tasks in nist_rmf_checklist.items():
    print(f"\n[{function}]")
    for task in tasks:
        print(f"  [ ] {task}")

NIST AI RMF Implementation Checklist

[GOVERN]
  [ ] AI risk management policies documented
  [ ] Accountability roles assigned (AI Officer or equivalent)
  [ ] Organizational AI risk tolerance defined
  [ ] Third-party AI system inventory maintained

[MAP]
  [ ] AI system purpose and context documented
  [ ] Risk classification completed (EU AI Act tier)
  [ ] Affected populations and potential harms identified
  [ ] Intended vs. actual deployment context verified

[MEASURE]
  [ ] Accuracy metrics benchmarked on representative data
  [ ] Demographic bias tests run on all protected attributes
  [ ] Adversarial robustness evaluated
  [ ] Human-AI team performance measured

[MANAGE]
  [ ] Automatic audit logging deployed
  [ ] Explainability wrapper operational
  [ ] Human override capability tested
  [ ] Incident response plan for AI failures
  [ ] Post-market monitoring dashboard live


## Section 5: Practical Compliance Workflow

The compliance workflow for a new AI system follows these steps:

1. **Classify** -- Determine risk tier using the RiskClassifier
2. **Document** -- Create technical documentation and model card
3. **Test** -- Run bias tests on training and validation data
4. **Instrument** -- Add AuditLogger and ExplainabilityWrapper
5. **Design** -- Add human oversight and override capability
6. **Register** -- Register in EU AI database (high-risk only)
7. **Monitor** -- Post-market monitoring for drift and new harms

The ComplianceReportGenerator below produces a structured report covering all steps.

In [8]:
class SystemMetrics(BaseModel):
    accuracy: float
    demographic_parity_gap: float
    logging_enabled: bool
    explainability_enabled: bool
    human_override_enabled: bool


class ComplianceReport(BaseModel):
    system_name: str
    system_version: str
    generated_at: str
    risk_tier: str
    checklist_completion: float
    metrics: SystemMetrics
    open_issues: List[str]
    deployment_approved: bool
    approver_notes: str


class ComplianceReportGenerator:
    """Generate a full EU AI Act compliance report for an AI system."""

    def generate(
        self,
        system_name: str,
        system_version: str,
        description: str,
        sector: str,
        use_case: str,
        metrics: SystemMetrics,
        checklist: ComplianceChecklist
    ) -> ComplianceReport:
        classifier = RiskClassifier()
        classification = classifier.classify_system(description, sector, use_case)

        open_issues = []

        if metrics.demographic_parity_gap > 0.10:
            open_issues.append(f"Bias: demographic parity gap {metrics.demographic_parity_gap:.2f} exceeds 0.10 threshold")

        if not metrics.logging_enabled:
            open_issues.append("Audit logging not yet deployed (Art. 12 requirement)")

        if not metrics.explainability_enabled:
            open_issues.append("Explainability wrapper not deployed (Art. 13 requirement)")

        if not metrics.human_override_enabled:
            open_issues.append("Human override capability not confirmed (Art. 14 requirement)")

        uncompleted = [item.requirement[:50] for item in checklist.items if not item.completed]
        for item_req in uncompleted:
            open_issues.append(f"Checklist incomplete: {item_req}...")

        approved = (
            len(open_issues) == 0
            and checklist.completion_rate() >= 1.0
            and metrics.accuracy >= 0.80
        )

        return ComplianceReport(
            system_name=system_name,
            system_version=system_version,
            generated_at=datetime.now().isoformat(),
            risk_tier=classification.risk_tier.value,
            checklist_completion=checklist.completion_rate(),
            metrics=metrics,
            open_issues=open_issues,
            deployment_approved=approved,
            approver_notes="Automated compliance check. Requires human AI Officer sign-off before deployment."
        )

    def print_report(self, report: ComplianceReport) -> None:
        print(f"EU AI ACT COMPLIANCE REPORT")
        print(f"System: {report.system_name} v{report.system_version}")
        print(f"Generated: {report.generated_at[:19]}")
        print(f"Risk Tier: {report.risk_tier.upper()}")
        print(f"Checklist Completion: {report.checklist_completion:.0%}")
        print(f"Accuracy: {report.metrics.accuracy:.1%}")
        print(f"Demographic Parity Gap: {report.metrics.demographic_parity_gap:.3f}")
        print(f"Logging Enabled: {report.metrics.logging_enabled}")
        print(f"Explainability Enabled: {report.metrics.explainability_enabled}")
        print(f"Human Override Enabled: {report.metrics.human_override_enabled}")
        print()
        if report.open_issues:
            print(f"OPEN ISSUES ({len(report.open_issues)}):")
            for issue in report.open_issues:
                print(f"  - {issue}")
        else:
            print("No open issues.")
        status = "APPROVED" if report.deployment_approved else "NOT APPROVED"
        print(f"\nDeployment Status: {status}")
        print(f"Notes: {report.approver_notes}")


# Demo: generate a report for our hiring screener
metrics = SystemMetrics(
    accuracy=0.87,
    demographic_parity_gap=0.23,  # Fails bias test
    logging_enabled=True,
    explainability_enabled=True,
    human_override_enabled=True
)

generator = ComplianceReportGenerator()
report = generator.generate(
    system_name="CVScreener",
    system_version="2.1.0",
    description="CV screening tool that ranks job applicants",
    sector="employment",
    use_case="hiring",
    metrics=metrics,
    checklist=checklist
)
generator.print_report(report)

EU AI ACT COMPLIANCE REPORT
System: CVScreener v2.1.0
Generated: 2026-06-29T12:19:09
Risk Tier: HIGH
Checklist Completion: 23%
Accuracy: 87.0%
Demographic Parity Gap: 0.230
Logging Enabled: True
Explainability Enabled: True
Human Override Enabled: True

OPEN ISSUES (11):
  - Bias: demographic parity gap 0.23 exceeds 0.10 threshold
  - Checklist incomplete: Risk management system: documented risk identifica...
  - Checklist incomplete: Data governance: training data documented, quality...
  - Checklist incomplete: Log retention: logs stored for minimum 10 years...
  - Checklist incomplete: Transparency: users notified they are interacting ...
  - Checklist incomplete: Explainability: explanations available for individ...
  - Checklist incomplete: Accuracy: performance metrics documented on repres...
  - Checklist incomplete: Robustness: adversarial robustness tested...
  - Checklist incomplete: Cybersecurity: model and API secured against adver...
  - Checklist incomplete: Conformity as

## Key Takeaways

1. **Classify first**: Before building anything, determine which EU AI Act risk tier your system falls into. Unacceptable-risk systems are banned; high-risk systems require significant engineering overhead.

2. **Compliance is engineering, not just paperwork**: The EU AI Act's high-risk obligations -- logging, explainability, bias testing, human oversight -- are implemented in code, not legal documents.

3. **Audit trails are mandatory**: Every AI decision affecting a person must be logged with timestamp, inputs (or their hash), outputs, and model version. Retention period is 10 years.

4. **Bias testing is a deployment gate**: Demographic parity testing must happen before deployment. A system that systematically disadvantages a protected group cannot be deployed as-is.

5. **Human oversight is non-negotiable for high-risk**: High-risk AI systems must be designed so a human can understand, monitor, and override every AI decision.

6. **NIST AI RMF and the EU AI Act align**: GOVERN, MAP, MEASURE, MANAGE maps directly onto the Act's requirements, giving engineers a practical engineering process for compliance.

7. **Phased implementation**: The Act's provisions roll out through 2026-2027. Banned systems: already prohibited. High-risk obligations: enforced from August 2026. Start compliance work now.